
# MNIST Digit Classification Project

**Name:** CHETAN H RAJPUT  
**Roll Number:** 251ME316  
**Branch:** Mechanical Engineering

## Project Objective
Build a Convolutional Neural Network (CNN) using PyTorch to classify handwritten digits (0–9) from the MNIST dataset.

This notebook covers:
- Dataset exploration
- Data preprocessing and normalization
- DataLoaders
- CNN architecture design
- Training loop
- Evaluation
- Model saving/loading
- Reflection and improvements



# Task 1 – Understand the Problem & Dataset

### What is MNIST?
MNIST is a dataset of handwritten digits from 0–9.

### Dataset Information
- Number of classes: 10
- Image size: 28 × 28 pixels
- Color format: Grayscale
- Training images: 60,000
- Test images: 10,000

### Input and Output
Input: Image tensor of shape `[1, 28, 28]`

Output: Tensor of shape `[10]` representing scores for digits 0–9.

### Why Normalize?
Normalization scales pixel values to a consistent range. This helps neural networks train faster and improves numerical stability.


In [ ]:

import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torchvision import datasets, transforms
from torch.utils.data import DataLoader


In [ ]:

raw_dataset = datasets.MNIST(
    root='data',
    train=True,
    download=True
)

print("Training Samples:", len(raw_dataset))
print("Image Shape:", raw_dataset.data[0].shape)
print("Label:", raw_dataset.targets[0].item())


In [ ]:

fig, axes = plt.subplots(2,5, figsize=(10,4))

for i, ax in enumerate(axes.flat):
    img, label = raw_dataset[i]
    ax.imshow(img, cmap='gray')
    ax.set_title(str(label))
    ax.axis('off')

plt.tight_layout()
plt.show()



# Task 2 – Data Pipeline

### DataLoader Responsibilities
- Batching
- Shuffling
- Efficient loading

### Transformations Used
1. ToTensor()
2. Normalize(mean=0.5, std=0.5)


In [ ]:

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST(
    root='data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root='data',
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

images, labels = next(iter(train_loader))

print("Images Shape:", images.shape)
print("Labels Shape:", labels.shape)



# Task 3 – CNN Architecture

Architecture:

Conv2D → ReLU → MaxPool  
Conv2D → ReLU → MaxPool  
Flatten → Dropout → Linear → ReLU → Linear

Dropout is used to reduce overfitting.


In [ ]:

class CNNNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.25),
            nn.Linear(64*7*7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = CNNNet()
print(model)


In [ ]:

total_params = sum(p.numel() for p in model.parameters())

print("Total Parameters:", total_params)

sample = torch.randn(32,1,28,28)
output = model(sample)

print("Output Shape:", output.shape)



# Task 4 – Training Loop


In [ ]:

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

loss_history = []

for epoch in range(5):

    running_loss = 0

    for images, labels in train_loader:

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)
    loss_history.append(epoch_loss)

    print(f"Epoch {epoch+1}: {epoch_loss:.4f}")


In [ ]:

plt.figure(figsize=(6,4))
plt.plot(loss_history)
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()



# Task 5 – Evaluation


In [ ]:

model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")



# Task 6 – Save, Load & Reflect


In [ ]:

torch.save(model.state_dict(), "mnist_cnn.pth")

loaded_model = CNNNet()
loaded_model.load_state_dict(torch.load("mnist_cnn.pth"))

print("Model saved and loaded successfully.")



## Reflection

### What I Learned
- How image data is represented as tensors.
- How DataLoaders simplify batching and shuffling.
- How CNNs extract features using convolution layers.
- How training and evaluation are performed in PyTorch.

### Possible Improvements
- Add more convolution layers.
- Experiment with learning rates.
- Train for more epochs.
- Use Batch Normalization.
- Apply data augmentation techniques.

### Conclusion
The CNN successfully learns meaningful features from handwritten digit images and achieves strong classification performance on the MNIST dataset.
